# RoadScan: train on Colab (free accounts, shared drive)

Fine-tunes **uzbtrust/smart-road-pci-yolo11** into a segmentation model for the Tunisian frames. This notebook only
**trains**: hard-negative mining (`tn.py mine`) and the demo videos (`demo_video.py`) run on the PC.

**On the shared drive `PersonalDrivePartage`, at the root:** `tn_data.zip` (from `python src/tn.py pack --extra ...`) and,
for `_r2` runs, `negatives.txt` (from the review page).

**Each time:** Runtime -> Change runtime type -> **T4 GPU**. Set `RUN` in cell 1. Runtime -> **Run all**.

| `RUN` | Base | Backbone | Negatives |
|---|---|---|---|
| `l640_freeze` | yolo11l, 640 | frozen (0-9) | no |
| `l640_freeze_r2` | yolo11l, 640 | frozen (0-9) | yes (`negatives.txt`) |
| `m1024`, `l640`, ... | m at 1024 / l at 640 | `_freeze` in the name = frozen | `_r2` in the name = yes |

- **Kicked out?** Same `RUN` on any other account, Run all: it continues from the last epoch.
- **Finished runs are never retrained** (a `DONE` file). To redo one from scratch, set `RESTART = True` once.
- Never run the same `RUN` on two accounts at once: they would write into the same folder.

In [ ]:
# 1. Settings: THE ONLY CELL TO EDIT
RUN = "l640_freeze_r2"
RESTART = False   # True = delete runs/<RUN> on the drive and train it again from scratch

SD = "/content/drive/Shareddrives/PersonalDrivePartage"
EPOCHS, PATIENCE = 100, 20
BASES = {"m1024": ("yolo11m-seg.yaml", "yolo11m_1024_best.pt", 1024),   # uzbtrust checkpoints on Hugging Face
         "l640": ("yolo11l-seg.yaml", "yolo11l_640_best.pt", 640)}
CFG, UZB_W, IMGSZ = BASES[RUN.split("_")[0]]
FREEZE = 10 if "_freeze" in RUN else None   # 10 = the YOLO11 backbone
USE_NEG = RUN.endswith("_r2")
print(f"RUN {RUN} | {CFG} at {IMGSZ} | freeze {FREEZE} | negatives {USE_NEG}")

In [ ]:
# 2. GPU check: expect a Tesla T4
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Data: mount the shared drive, unpack to the local disk, map our classes onto uzbtrust's, add the negatives
from google.colab import drive
drive.mount("/content/drive")
!pip -q install ultralytics==8.4.155

import shutil, zipfile, yaml
from pathlib import Path
TN = Path("/content/tn")
shutil.rmtree(TN, ignore_errors=True)   # a fresh copy every time, so the remap below runs exactly once
zipfile.ZipFile(f"{SD}/tn_data.zip").extractall(TN)

# our 5 classes -> uzbtrust's 10 slots, so its trained box/class head carries over (slots 8, 9 were never trained)
REMAP = {0: 4, 1: 1, 2: 7, 3: 8, 4: 9}   # pothole, crack (our cracks are mesh = alligator), manhole, trash, other
NAMES = ["lt_crack", "crack", "block_crack", "patching", "pothole", "raveling", "dropoff", "manhole", "trash", "other"]
for f in (TN / "labels").glob("*.txt"):
    f.write_text("".join(f"{REMAP[int(l.split()[0])]} {l.split(' ', 1)[1]}" for l in f.read_text().splitlines(True) if l.strip()))

train = (TN / "train.txt").read_text().split()
if USE_NEG:   # frames where every prediction was wrong become background (empty label)
    neg = Path(f"{SD}/negatives.txt").read_text().split()
    missing = [n for n in neg if not (TN / "images" / n).exists()]
    assert neg, "negatives.txt on the drive is EMPTY: upload data/negatives.txt from the PC again"
    assert not missing, f"{len(missing)} negatives are not in tn_data.zip (e.g. {missing[0]}): run tn.py pack --extra ... again"
    for n in neg:
        (TN / "labels" / f"{Path(n).stem}.txt").write_text("")
    train += [f"./images/{n}" for n in neg]
(TN / "train_run.txt").write_text("\n".join(train) + "\n")
yaml.safe_dump({"path": str(TN), "train": "train_run.txt", "val": "val.txt", "names": dict(enumerate(NAMES))},
               open(TN / "data.yaml", "w"))
print(f"RUN {RUN} | {len(train)} train frames ({len(train) - len((TN / 'train.txt').read_text().split())} negatives), "
      f"{len((TN / 'val.txt').read_text().split())} val frames")

In [ ]:
# 4. Train, or resume after a kick-out (from any account), or skip if finished
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

RUN_DIR = Path(f"{SD}/runs/{RUN}")
if RESTART and RUN_DIR.exists():
    shutil.rmtree(RUN_DIR); print("deleted", RUN_DIR, "- set RESTART = False again before the next Run all")
if (RUN_DIR / "DONE").exists():   # needed: resume=True on a finished run silently starts over from scratch
    print(f"{RUN} already finished: weights in {RUN_DIR}/weights (RESTART = True to train it again)")
elif (RUN_DIR / "weights" / "last.pt").exists():
    print("resuming", RUN)
    YOLO(str(RUN_DIR / "weights" / "last.pt")).train(resume=True)
    (RUN_DIR / "DONE").touch()
else:
    w = hf_hub_download("uzbtrust/smart-road-pci-yolo11", UZB_W, local_dir=f"{SD}/weights")
    # pretrained= loads straight into the 10-class seg model ("Transferred N/M items" in the log; the missing items
    # are the new mask branch). YOLO(CFG).load(w) would go through an 80-class model and lose the class head.
    YOLO(CFG).train(pretrained=w, data=str(TN / "data.yaml"), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=0.7,
                    freeze=FREEZE,
                    degrees=20, perspective=0.0005, shear=5,   # tilted / rotated phone
                    scale=0.5, hsv_v=0.4, hsv_s=0.35,           # size and intensity (hsv_s 0.35 won tn5)
                    copy_paste=0.5,                             # pasted masks get the same rotation/scale as the image
                    cache="ram", workers=2, plots=False, project=f"{SD}/runs", name=RUN, exist_ok=True)
    shutil.copy(TN / "train_run.txt", RUN_DIR / "train_run.txt")   # record of exactly which frames it trained on
    (RUN_DIR / "DONE").touch()
print("weights:", sorted(p.name for p in (RUN_DIR / "weights").glob("*.pt")))

### Back on the PC
Download `PersonalDrivePartage/runs/<RUN>/` into `runs/<RUN>/`, then judge it on the hold-out videos (never used as negatives):

```
python src/demo_video.py --weights runs/<RUN>/weights/best.pt --video data/raw/tn/20260926_130704.mp4 --device 0 --fps 6 --show pothole crack --roi 0.28 0.61 --persist 3
python src/demo_video.py --weights runs/<RUN>/weights/best.pt --video data/raw/tn/20260926_080755.mp4 --device 0 --fps 6 --show pothole crack --roi 0.30 0.80 --persist 3
```

The model uses uzbtrust's 10 class slots: pothole = 4, crack = 1, manhole = 7, trash = 8, other = 9.